In [32]:
url = "https://servicodados.ibge.gov.br/api/v3/agregados/4093/periodos/201201|201202|201203|201204|201301|201302|201303|201304|201401|201402|201403|201404|201501|201502|201503|201504|201601|201602|201603|201604|201701|201702|201703|201704|201801|201802|201803|201804|201901|201902|201903|201904|202001|202002|202003|202004|202101|202102|202103|202104|202201|202202|202203|202204|202301|202302|202303|202304|202401|202402|202403|202404|202501|202502|202503|202504/variaveis/4099?localidades=N3[26]&classificacao=2[6794]"
urlHomens = "https://servicodados.ibge.gov.br/api/v3/agregados/4093/periodos/201201|201202|201203|201204|201301|201302|201303|201304|201401|201402|201403|201404|201501|201502|201503|201504|201601|201602|201603|201604|201701|201702|201703|201704|201801|201802|201803|201804|201901|201902|201903|201904|202001|202002|202003|202004|202101|202102|202103|202104|202201|202202|202203|202204|202301|202302|202303|202304|202401|202402|202403|202404|202501|202502|202503|202504/variaveis/4099?localidades=N3[26]&classificacao=2[4]"
urlMulheres = "https://servicodados.ibge.gov.br/api/v3/agregados/4093/periodos/201201|201202|201203|201204|201301|201302|201303|201304|201401|201402|201403|201404|201501|201502|201503|201504|201601|201602|201603|201604|201701|201702|201703|201704|201801|201802|201803|201804|201901|201902|201903|201904|202001|202002|202003|202004|202101|202102|202103|202104|202201|202202|202203|202204|202301|202302|202303|202304|202401|202402|202403|202404|202501|202502|202503|202504/variaveis/4099?localidades=N3[26]&classificacao=2[5]"

In [33]:
import pandas as pd
data = pd.read_json(url)

dataHomens = pd.read_json(urlHomens)
dataMulheres = pd.read_json(urlMulheres)

In [34]:
serie = data['resultados'][0][0]['series'][0]['serie']

In [35]:
df = pd.DataFrame.from_dict(serie, orient='index', columns=['valor'])
df.index.name = 'periodo'
df = df.reset_index() 

In [36]:
df.head()

,periodo,valor
0,201201,9.6
1,201202,8.3
2,201203,9.4
3,201204,9.2
4,201301,10.7


In [37]:
df = df.replace('...', '0')

### Transformar [valor em FLOAT]

In [38]:
df['valor'] = df['valor'].astype('float')

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   periodo  56 non-null     str    
 1   valor    56 non-null     float64
dtypes: float64(1), str(1)
memory usage: 1.0 KB


In [40]:
df.head()

,periodo,valor
0,201201,9.6
1,201202,8.3
2,201203,9.4
3,201204,9.2
4,201301,10.7


### Periodo tem que virar data

In [41]:
df['ano'] = df['periodo'].str[:4].astype('int')

In [42]:
df['trimestre'] = df['periodo'].str[-2:].astype('int')

In [43]:
df['periodo'] = pd.PeriodIndex(df['ano'].astype('str') + 'Q' + df['trimestre'].astype('str'), freq='Q')

In [44]:
df['periodo'] = df['periodo'].dt.to_timestamp()

In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   periodo    56 non-null     datetime64[us]
 1   valor      56 non-null     float64       
 2   ano        56 non-null     int64         
 3   trimestre  56 non-null     int64         
dtypes: datetime64[us](1), float64(1), int64(2)
memory usage: 1.9 KB


In [46]:
df.head()

,periodo,valor,ano,trimestre
0,2012-01-01,9.6,2012,1
1,2012-04-01,8.3,2012,2
2,2012-07-01,9.4,2012,3
3,2012-10-01,9.2,2012,4
4,2013-01-01,10.7,2013,1


### Extrair dados para a taxa de desocupação em Pernambuco, para Homens e Mulheres e depois juntar os dados tratados em único dataframe e salvar o resultado em um CSV

#### Homens

In [47]:
dataHomens.head()

,id,variavel,unidade,resultados
0,4099,"Taxa de desocupação, na semana de referência, ...",%,"[{'classificacoes': [{'id': '2', 'nome': 'Sexo..."


In [48]:
serieHomens = dataHomens['resultados'][0][0]['series'][0]['serie']

In [49]:
df1 = pd.DataFrame.from_dict(serieHomens, orient='index', columns=['valor'])
df1.index.name = 'periodo'
df1 = df1.reset_index()

In [50]:
df1.head()

,periodo,valor
0,201201,7.1
1,201202,6.4
2,201203,7.6
3,201204,7.9
4,201301,8.2


In [51]:
df1 = df1.replace('...', '0')

In [52]:
df1['valor'] = df1['valor'].astype('float')

In [53]:
df1 = df.replace('...', '0')

In [54]:
df1['periodo'] = pd.to_datetime(df1['periodo'])

In [55]:
df1['ano'] = df1['periodo'].dt.year

In [56]:
df1.head()

,periodo,valor,ano,trimestre
0,2012-01-01,9.6,2012,1
1,2012-04-01,8.3,2012,2
2,2012-07-01,9.4,2012,3
3,2012-10-01,9.2,2012,4
4,2013-01-01,10.7,2013,1


In [57]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   periodo    56 non-null     datetime64[us]
 1   valor      56 non-null     float64       
 2   ano        56 non-null     int32         
 3   trimestre  56 non-null     int64         
dtypes: datetime64[us](1), float64(1), int32(1), int64(1)
memory usage: 1.7 KB


#### Mulheres

In [58]:
dataMulheres.head()

,id,variavel,unidade,resultados
0,4099,"Taxa de desocupação, na semana de referência, ...",%,"[{'classificacoes': [{'id': '2', 'nome': 'Sexo..."


In [59]:
serieMulheres = dataMulheres['resultados'][0][0]['series'][0]['serie']

In [60]:
df2 = pd.DataFrame.from_dict(serieHomens, orient='index', columns=['valor'])
df2.index.name = 'periodo'
df2 = df2.reset_index()

In [61]:
df2 = df.replace('...', '0')

In [62]:
df2['valor'] = df2['valor'].astype('float')

In [63]:
df2['periodo'] = pd.to_datetime(df2['periodo'])

In [64]:
df2['ano'] = df2['periodo'].dt.year

In [65]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   periodo    56 non-null     datetime64[us]
 1   valor      56 non-null     float64       
 2   ano        56 non-null     int32         
 3   trimestre  56 non-null     int64         
dtypes: datetime64[us](1), float64(1), int32(1), int64(1)
memory usage: 1.7 KB


### Merge

In [66]:
dataFrame_final = pd.merge(df1, df2, on='periodo', suffixes=('_homens', '_mulheres'))

#### Salvar em um CSV

In [67]:
dataFrame_final.to_csv('dataFrame_final.csv', index=False, encoding='utf-8', sep=';')